# MobileNetV2 전이학습 (SP3 직접 학습 모델)

Teachable Machine 대신 Keras로 직접 학습해 `.tflite`를 만든다. 결과 파일은 `code/` 의 `model_unquant.tflite`, `labels.txt` 를 그대로 바꿔 끼우면 된다.

**준비:** 파이에서 `capture.py` 로 찍은 `data/` 폴더를 zip으로 묶는다. 안에 `plastic/`, `can/`, `general/`, `empty/` 폴더가 있어야 한다. 판정 수정된 사진은 `corrected/<품목>/` 에 넣으면 같이 학습한다.

**실행:** 런타임 → 런타임 유형 변경 → GPU(T4) → 위에서부터 차례로 실행.

In [ ]:
# 1. data.zip 올리기 (왼쪽 파일 탭에 끌어다 놓아도 됨)
from google.colab import files
up = files.upload()            # data.zip 선택
!rm -rf data && unzip -q -o data.zip -d . && ls data

In [ ]:
# 2. 설정
import os, shutil, numpy as np, tensorflow as tf
CLASSES = ["plastic", "can", "general", "empty"]   # labels.txt 순서 = 모델 출력 순서
IMG = 224
BATCH = 32
SEED = 42

# 판정 수정 사진(corrected/<품목>/*.jpg)이 있으면 data/<품목>/ 에 합친다
if os.path.isdir("data/corrected"):
    for c in os.listdir("data/corrected"):
        for f in os.listdir(f"data/corrected/{c}"):
            shutil.copy(f"data/corrected/{c}/{f}", f"data/{c}/corrected_{f}")
    shutil.rmtree("data/corrected")
for c in CLASSES:
    print(c, len(os.listdir(f"data/{c}")), "장")

In [ ]:
# 3. 학습 80% / 시험 20% 나누기
train_ds, test_ds = tf.keras.utils.image_dataset_from_directory(
    "data", class_names=CLASSES, image_size=(IMG, IMG), batch_size=BATCH,
    validation_split=0.2, subset="both", seed=SEED, label_mode="int")
AUTO = tf.data.AUTOTUNE
train_ds, test_ds = train_ds.prefetch(AUTO), test_ds.prefetch(AUTO)

In [ ]:
# 4. 모델: MobileNetV2(ImageNet) 특징 추출부 고정 + 분류층 4개
# 입력 전처리는 main.py 와 같게: 가운데 정사각형 224x224, 픽셀/127.5 - 1  (모델 안에 넣지 않음)
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomBrightness(0.2, value_range=(-1, 1)),
    tf.keras.layers.RandomZoom(0.1),
], name="augment")

base = tf.keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet")
base.trainable = False

inputs = tf.keras.Input((IMG, IMG, 3))
x = augment(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(CLASSES), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

scale = lambda x, y: (x / 127.5 - 1.0, y)
train_s, test_s = train_ds.map(scale), test_ds.map(scale)

In [ ]:
# 5. 1단계 학습 (분류층만)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h1 = model.fit(train_s, validation_data=test_s, epochs=10)

In [ ]:
# 6. 2단계 미세 조정 (특징 추출부 위쪽 30층만 풀고 작은 학습률)
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = model.fit(train_s, validation_data=test_s, epochs=10)

In [ ]:
# 7. 시험 사진 20%로 평가: 정확도 + 혼동행렬 (기획서 7-4 "모델 정확도")
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
y_true = np.concatenate([y.numpy() for _, y in test_s])
y_pred = np.argmax(model.predict(test_s), axis=1)
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))
items = [i for i, c in enumerate(CLASSES) if c != "empty"]   # 목표: 품목 3종 정확도 85% 이상
mask = np.isin(y_true, items)
print("품목 3종 정확도:", round(float((y_true[mask] == y_pred[mask]).mean()) * 100, 1), "%")
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASSES).plot(cmap="Blues")
plt.title("혼동행렬 (시험 사진 20%)"); plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# 8. TFLite 로 변환하고 내려받기 → code/ 폴더의 같은 이름 파일을 바꿔 끼운다
conv = tf.lite.TFLiteConverter.from_keras_model(model)
open("model_unquant.tflite", "wb").write(conv.convert())
with open("labels.txt", "w") as f:
    for i, c in enumerate(CLASSES):
        f.write(f"{i} {c}
")
print(os.path.getsize("model_unquant.tflite") // 1024, "KB")
files.download("model_unquant.tflite"); files.download("labels.txt"); files.download("confusion_matrix.png")